### ***Libraries Used***

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

RANDOM_STATE = 42

print("Libraries imported successfully!")


### **Load the Dataset from Github**

In [ ]:


URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"

# column names
COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
    'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty'
]

# train data
train = pd.read_csv(
    URL + "KDDTrain%2B.txt",
    header=None,
    names=COLUMNS
)

#test data
test = pd.read_csv(
    URL + "KDDTest%2B.txt",
    header=None,
    names=COLUMNS
)

print("Training Dataset:", train.shape)
print("Testing Dataset:", test.shape)


### **To display the first 5 rows**

In [ ]:
train.head()

In [ ]:
print("Number of rows:", train.shape[0])
print("Number of columns:", train.shape[1])

train.info()


In [ ]:

train.describe()


### **Attack** **types**

In [ ]:

print(train['label'].value_counts())


In [ ]:

plt.figure(figsize=(14, 6))

train['label'].value_counts().plot(
    kind='bar',
    color='teal'
)

plt.title('Distribution of Network Traffic Classes')
plt.xlabel('Traffic Type')
plt.ylabel('Number of Records')
plt.xticks(rotation=90)

plt.show()


Khleha delw2ty

In [ ]:

# Define attack families

DOS = {
    'back', 'land', 'neptune', 'pod', 'smurf',
    'teardrop', 'apache2', 'udpstorm',
    'processtable', 'worm', 'mailbomb'
}

PROBE = {
    'satan', 'ipsweep', 'nmap',
    'portsweep', 'mscan', 'saint'
}

U2R = {
    'buffer_overflow', 'loadmodule',
    'rootkit', 'perl', 'sqlattack',
    'xterm', 'ps'
}

def family(label):

    if label == 'normal':
        return 'Normal'

    elif label in DOS:
        return 'DoS'

    elif label in PROBE:
        return 'Probe'

    elif label in U2R:
        return 'U2R'

    else:
        return 'R2L'

# Create multiclass target
train['family'] = train['label'].map(family)
test['family'] = test['label'].map(family)

# Create binary target
train['is_attack'] = (train['label'] != 'normal').astype(int)
test['is_attack'] = (test['label'] != 'normal').astype(int)

print(train[['label', 'family', 'is_attack']].head(10))


### **Fetures Selection**

In [ ]:

FEATURES = [
    'duration',
    'src_bytes',
    'dst_bytes',
    'num_failed_logins',
    'count',
    'srv_count',
    'serror_rate',
    'same_srv_rate'
]

X = train[FEATURES]

y_binary = train['is_attack']
y_multi = train['family']

print("Selected Features:")
print(X.head())

print("Number of Features:", len(FEATURES))


### **Data Preprocessing**

In [ ]:

missing_values = train.isnull().sum()

print(missing_values[missing_values > 0])

print("Total missing values:", train.isnull().sum().sum())


In [ ]:

print("Duplicates before cleaning:", train.duplicated().sum())

train = train.drop_duplicates().copy()

print("Duplicates after cleaning:", train.duplicated().sum())


In [ ]:


X = train[FEATURES].copy()

y_binary = train['is_attack'].copy()
y_multi = train['family'].copy()

print("Input Features:", X.columns.tolist())
print("Target columns are kept separately.")


In [ ]:


numerical_columns = X.select_dtypes(
    include=[np.number]
).columns

# Calculate IQR

Q1 = X[numerical_columns].quantile(0.25)
Q3 = X[numerical_columns].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Count outliers

for col in numerical_columns:

    outliers = (
        (X[col] < lower_bound[col]) |
        (X[col] > upper_bound[col])
    )

    print(col, "Outliers:", outliers.sum())


### **Train/Validate/Split**

In [ ]:

# Split the original training data

X_train, X_val, y_binary_train, y_binary_val, y_multi_train, y_multi_val = train_test_split(

    X,
    y_binary,
    y_multi,

    test_size=0.20,
    random_state=42,

    # Preserve class distribution
    stratify=y_multi
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Official test samples:", len(test))

print("\nTraining class distribution:")
print(y_multi_train.value_counts())


### **Class Imbalance Handling**

In [ ]:

# Check class distribution

print("Before handling imbalance:")

print(y_binary_train.value_counts())

print("\nClass proportions:")

print(y_binary_train.value_counts(normalize=True))


In [ ]:

plt.figure(figsize=(8, 5))

sns.countplot(
    x=y_multi_train,
    order=y_multi_train.value_counts().index
)

plt.title("Multiclass Class Distribution")
plt.xlabel("Attack Family")
plt.ylabel("Number of Records")

plt.show()


### **Baseline Model**

 **(Binary Classification)**

In [ ]:

# Build Binary Classification Pipeline

binary_model = Pipeline([

    ('scaler', StandardScaler()),

    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))

])

# Train the model

start_time = time.perf_counter()

binary_model.fit(X_train, y_binary_train)

fit_time = time.perf_counter() - start_time

print("Binary model trained successfully!")
print("Training time:", fit_time)


### **Predictions Binary**

In [ ]:

# Predict on validation data

start_time = time.perf_counter()

binary_predictions = binary_model.predict(X_val)

prediction_time = time.perf_counter() - start_time

print("Predictions completed!")

print("Prediction time:", prediction_time)


### **Evaluate Binary**

In [ ]:

# Classification Report

print(classification_report(
    y_binary_val,
    binary_predictions,
    target_names=['Normal', 'Attack'],
    digits=4
))


### **Confusion Matrix Binary**

In [ ]:

# Create Confusion Matrix

cm = confusion_matrix(
    y_binary_val,
    binary_predictions
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Normal', 'Attack']
)

display.plot(cmap='Blues', values_format='d')

plt.title("Binary Classification - Confusion Matrix")

plt.show()


### **Baseline**

**(Multiclass)**

In [ ]:

# Build Multiclass Logistic Regression

multi_model = Pipeline([

    ('scaler', StandardScaler()),

    ('classifier', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))

])

# Train

multi_model.fit(X_train, y_multi_train)

print("Multiclass model trained successfully!")


### **Predications Multiclass**

In [ ]:

# Multiclass predictions

multi_predictions = multi_model.predict(X_val)

print(classification_report(
    y_multi_val,
    multi_predictions,
    digits=4,
    zero_division=0
))


### **Confusion Matrix Multiclass**

In [ ]:

# Multiclass Confusion Matrix

labels = ['Normal', 'DoS', 'Probe', 'U2R', 'R2L']

cm_multi = confusion_matrix(
    y_multi_val,
    multi_predictions,
    labels=labels
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm_multi,
    display_labels=labels
)

display.plot(cmap='Blues', xticks_rotation=45)

plt.title("Multiclass Classification")
plt.show()


### **Per-class Recall**

In [ ]:

# Calculate recall for each class

per_class_recall = recall_score(
    y_multi_val,
    multi_predictions,
    labels=labels,
    average=None,
    zero_division=0
)

recall_table = pd.DataFrame({

    'Attack Family': labels,

    'Recall': per_class_recall

})

print(recall_table)


### **Leakage Check**

In [ ]:

# Leakage checks

print("Target leakage check:")

for col in ['label', 'difficulty', 'family', 'is_attack']:

    print(col, "in X:", col in X.columns)


print("\nTraining and validation overlap:")

print(
    "Same indices:",
    len(set(X_train.index) & set(X_val.index))
)


print("\nFeature columns:")

print(X_train.columns.tolist())


# Check scaler fitted on training data only

print("\nScaler mean:")

print(binary_model.named_steps['scaler'].mean_[:5])
